In [1]:
import io
import pandas as pd
import numpy as np

# Sample dirty dataset represented as a string
raw_csv_data = """Name , Age , Salary , Joining Date , City
Alice, 25, 50000, 2021-05-10, New York
Bob, , 60000, 2020-01-15, London
Alice, 25, 50000, 2021-05-10, New York
Charlie, 30, , 2019-11-20, Paris
David, 45, 120000, 2018-07-30, 
Eva, 200, 45000, invalid_date, London
"""

def clean_and_preprocess_data(csv_stream):
    # 1. Load Dataset
    df = pd.read_csv(csv_stream)
    print("--- Initial Data ---")
    print(df)
    
    # 2. Standardize Column Headers
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # 3. Remove Duplicate Rows
    df = df.drop_duplicates().reset_index(drop=True)

    # 4. Handle String Inconsistencies & Whitespace
    object_cols = df.select_dtypes(include=['object']).columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].replace({'nan': np.nan, 'none': np.nan, 'null': np.nan, '': np.nan})

    # 5. Correct Data Types
    for col in df.columns:
        if 'date' in col or 'time' in col:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif col in ['age', 'salary', 'price', 'amount']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 6. Handle Missing Values
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')

    date_cols = df.select_dtypes(include=['datetime64[ns]']).columns
    for col in date_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].ffill()

    # 7. Outlier Handling (IQR Method)
    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df[col] = np.clip(df[col], lower_bound, upper_bound)

    print("\n--- Cleaned Data ---")
    print(df)
    return df

if __name__ == "__main__":
    clean_and_preprocess_data(io.StringIO(raw_csv_data))

--- Initial Data ---
     Name   Age   Salary   Joining Date        City
0    Alice    25    50000     2021-05-10   New York
1      Bob          60000     2020-01-15     London
2    Alice    25    50000     2021-05-10   New York
3  Charlie    30              2019-11-20      Paris
4    David    45   120000     2018-07-30           
5      Eva   200    45000   invalid_date     London

--- Cleaned Data ---
      name   age   salary joining_date      city
0    alice  25.0  50000.0   2021-05-10  new york
1      bob  37.5  60000.0   2020-01-15    london
2  charlie  30.0  55000.0   2019-11-20     paris
3    david  45.0  75000.0   2018-07-30    london
4      eva  67.5  45000.0   2018-07-30    london


In [2]:
# Function se cleaned dataframe leke CSV me export karein
cleaned_df = clean_and_preprocess_data(io.StringIO(raw_csv_data))
cleaned_df.to_csv("cleaned_data.csv", index=False)
print("File 'cleaned_data.csv' successfully saved!")

--- Initial Data ---
     Name   Age   Salary   Joining Date        City
0    Alice    25    50000     2021-05-10   New York
1      Bob          60000     2020-01-15     London
2    Alice    25    50000     2021-05-10   New York
3  Charlie    30              2019-11-20      Paris
4    David    45   120000     2018-07-30           
5      Eva   200    45000   invalid_date     London

--- Cleaned Data ---
      name   age   salary joining_date      city
0    alice  25.0  50000.0   2021-05-10  new york
1      bob  37.5  60000.0   2020-01-15    london
2  charlie  30.0  55000.0   2019-11-20     paris
3    david  45.0  75000.0   2018-07-30    london
4      eva  67.5  45000.0   2018-07-30    london
File 'cleaned_data.csv' successfully saved!
